In [6]:
%load_ext autoreload
%autoreload 2
    
%matplotlib notebook

import numpy as np
from pprint import pprint

# Overview

We will begin by initializing a full example of a `SEAFile` so that we can walk through some of the basic features that will be common place in package. Below is a bare bones example of a `SEAFile` hierarchy built from the ground up. Most readers will handle this for you and there are typically multiple ways to manually instantiate any of these classes, which will be described in subsequent sections. Here, I am reusing a single dataset and only instantiate the `SEAFile.Experiment` for brevity.

In [8]:
import sys
sys.path.insert(1,"../src")
from pySEA.sea_eco.architecture.base_structure_numpy import Metadata, Dimensions, Signal, SignalSet, SignalCollection, SEAFile
from pySEA.sea_eco.test_dataset_generators import generate_gaussian_SI_t_series_data

# Create a Metadata object.
# Note that a dict could likewise be passed directly to most SEAs objects.
meta_dict = {'General': {'title': 'Some Title', 
                         'uuid': 'some UUID'}, 
             'Instrument': {'beam_energy':60E3, 
                            'Detectors': {'Dectris_ELA': {'name':'Dectris ELA','exposure':0.01}}, 
                            'Scan': {'uuid': 'XXXXXXXX0000000', 'scan_rotation':0, 'dwell_time':4.0},
                            },
             'some meta': 1.}
meta = Metadata(meta_dict)

# Difne the dimensions.
#Note that a dict could likewise be passed directly to most SEAs objects.
kwarg_dict0 = dict(name='t', space='temporal', units='frame', size=5, offset=0, scale=1)
kwarg_dict1 = dict(name='y', space='position', units='nm', size=11, offset=-60, scale=10)
kwarg_dict2 = dict(name='x', space='position', units='nm', size=21, offset=-60, scale=5)
kwarg_dict3 = dict(name='E', space='spectral', units='eV',size=512, offset=-1.6, scale=0.1)
kwarg_dicts = [kwarg_dict0, kwarg_dict1, kwarg_dict2, kwarg_dict3]
dimensions = Dimensions(kwarg_dicts, nav_dimensions=[1,2], sig_dimensions=[3])

# Generate the signal and data
sig_data = generate_gaussian_SI_t_series_data(dimensions=dimensions)
signal = Signal(sig_data, name='SI', dimensions=dimensions, metadata=meta_dict,
                 original_metadata=Metadata({'source':'simulated data'})
                )
adf = signal.deepcopy_with_reduced_data_dim(-0.1*np.sum(signal[...,:5.], axis=-1), keep_dim=(0,1,2))
adf.metadata.Instrument.Detectors.update_from_dict({'ADF':{'exposure':10.}})
del adf.metadata.Instrument.Detectors.Dectris_ELA

abf = signal.deepcopy_with_reduced_data_dim(np.sum(signal[...,:5.], axis=-1), keep_dim=(0,1,2))
abf.metadata.Instrument.Detectors.update_from_dict({'ABF':{'exposure':10.}})
del abf.metadata.Instrument.Detectors.Dectris_ELA

#Generate the higher level signal set and collection objects
signals = [adf, abf, signal]
names = ['ADF', 'vBF', 'SI']
for signal,name in zip(signals,names): signal.name=name
signal_set = SignalSet(signals=signals, name='SI scan 1', main_signal=-1,metadata={'key':'set level metadata'})


signal_collection = SignalCollection(datasets=[signal, signal_set], metadata={'key':'collection level metadata'})

sea_file = SEAFile(experiments=signal_collection, metadata={'key':'file level metadata'})

sea_file.show_tree()

## Attribute Search

In [9]:
print('uuids in sea_file:')
pprint(sea_file.find_attribute('uuid', include_hidden=False))
print()

print('names in signal set:')
pprint(signal_set.find_attribute('name', include_hidden=False))

uuids in sea_file:
{'Experiments.datasets[0].metadata.General.uuid': 'some UUID',
 'Experiments.datasets[0].metadata.Instrument.Scan.uuid': 'XXXXXXXX0000000',
 'Experiments.datasets[0].uuid': '387a81f3-2aea-4618-859b-9e21f7f7de52',
 'Experiments.datasets[1].datasets[0].metadata.General.uuid': 'some UUID',
 'Experiments.datasets[1].datasets[0].metadata.Instrument.Scan.uuid': 'XXXXXXXX0000000',
 'Experiments.datasets[1].datasets[0].uuid': 'e6a70482-a849-48f6-b2f7-7406ac511673',
 'Experiments.datasets[1].datasets[1].metadata.General.uuid': 'some UUID',
 'Experiments.datasets[1].datasets[1].metadata.Instrument.Scan.uuid': 'XXXXXXXX0000000',
 'Experiments.datasets[1].datasets[1].uuid': 'a5927cba-7ac9-4475-9a1a-033377264262',
 'Experiments.datasets[1].datasets[2].metadata.General.uuid': 'some UUID',
 'Experiments.datasets[1].datasets[2].metadata.Instrument.Scan.uuid': 'XXXXXXXX0000000',
 'Experiments.datasets[1].datasets[2].uuid': '1cdf7b84-a0ae-49dc-b0b3-9758aa3f3dfa',
 'Experiments.dataset

# Metadata class
Check that metadata can be read from all of the test files and converted to the GeneralMetada class

In [10]:
from pySEA.sea_eco.architecture.base_structure_numpy import Metadata

Metadata()

In [11]:
meta_dict = {'General': {'title': 'Some Title', 
                         'uuid': 'some UUID'}, 
             'Instrument': {'beam_energy':60E3, 
                            'Detectors': {'Dectris_ELA': {'name':'Dectris ELA','exposure':0.01}}, 
                            'Scan': {'scan_uuid': 'XXXXXXXX0000000', 'scan_rotation':0, 'dwell_time':4.0},
                            },
             'some meta': 1.}
meta = Metadata(meta_dict)
meta

├── General
|  ├── title: Some Title
|  └── uuid: some UUID
├── Instrument
|  ├── beam_energy: 60000.0
|  ├── Detectors
|  |  └── Dectris_ELA
|  |     ├── name: Dectris ELA
|  |     └── exposure: 0.01
|  └── Scan
|     ├── scan_uuid: XXXXXXXX0000000
|     ├── scan_rotation: 0
|     └── dwell_time: 4.0
└── some meta: 1.0

# Dimensions and Dimension Class
Demonstrate Dimension creation from kwargs. <br />
Demonstrate Dimensions and Dimension handling from metadata

In [12]:
from pySEA.sea_eco.architecture.base_structure_numpy import Dimension, Dimensions

Dimension().show_tree()
Dimensions().show_tree()

/media/qwe/Alexandria/ORNL/Various Code/multislice/PySlice/dev/tests/sea-eco/examples/../src/pySEA/sea_eco/architecture/base_structure_numpy.py:924: UserWarning: size is None.
  if key not in checks.items(): warn(f'{key} is None.')
/media/qwe/Alexandria/ORNL/Various Code/multislice/PySlice/dev/tests/sea-eco/examples/../src/pySEA/sea_eco/architecture/base_structure_numpy.py:924: UserWarning: offset is None.
  if key not in checks.items(): warn(f'{key} is None.')
/media/qwe/Alexandria/ORNL/Various Code/multislice/PySlice/dev/tests/sea-eco/examples/../src/pySEA/sea_eco/architecture/base_structure_numpy.py:924: UserWarning: scale is None.
  if key not in checks.items(): warn(f'{key} is None.')


## Creating dimension from kwargs

Creation of an 1D-dimension from atributes

In [ ]:
kwarg_dict = dict(name='x', space='position', units='nm',
                  size=5, offset=-10, scale=10)
dimension = Dimension(**kwarg_dict)

print(f'Dict:            {kwarg_dict}')
print(f'Dimension (str):      {dimension}')
print(f'Dimension (__repr__): {dimension.__repr__()}')
print(f'Shape:           {dimension.values.shape}')
dimension.show_tree()


In [ ]:
kwarg_dict = dict(name='xy', space='position', units='nm',
                  size=5, offset=[-1,-10], scale=[1,10])
dimension = Dimension(**kwarg_dict)
print(f'Dict:                 {kwarg_dict}')
print(f'Dimension (str):      {dimension}')
print(f'Dimension (__repr__): {dimension.__repr__()}')
print(f'Shape:                {dimension.values.shape}')
dimension.show_tree()


In [ ]:
array = np.array([np.arange(11),np.arange(11)* 10])
kwarg_dict = dict(name=['x','y'], space='position', units='nm',
                  values=array)

dimension = Dimension(**kwarg_dict)
print(f'Dict:')
pprint(kwarg_dict, indent=5)
print(f'Dimension (str):      {dimension}')
print(f'Dimension (__repr__): {dimension.__repr__()}')
print(f'Shape:                {dimension.values.shape}')
dimension.show_tree()

## Creating Dimensions from kwargs

The Dimension class can be created by providing a list of dictionarys representing the dimensions

In [ ]:
kwarg_dict0 = dict(name='t', space='temporal', units='frame',
                  size=5, offset=0, scale=1)
kwarg_dict1 = dict(name='y', space='position', units='nm',
                  size=11, offset=-60, scale=10)
kwarg_dict2 = dict(name='x', space='position', units='nm',
                  size=21, offset=-60, scale=5)
kwarg_dict3 = dict(name='E', space='spectral', units='eV',
                  size=512, offset=-1.6, scale=0.1)

kwarg_dicts = [kwarg_dict0, kwarg_dict1, kwarg_dict2, kwarg_dict3]
dimensions = Dimensions(kwarg_dicts)

dimensions.show_tree()

The Dimensions class can also be initialized by supplying dimensions

In [ ]:
dimensions = Dimensions([Dimension(kws) for kws in kwarg_dicts], nav_dimensions=[1,2], sig_dimensions=[3])
dimensions.show_tree()

Dimensions can also be added after initialization using `dimensions.add_dimension()`. This is helpful for creating and modifying Dimensions when designing analysis workflows. However, one should be carefull as this could create mismatch with data dimensions within a class.

In [ ]:
#FLAG delete this cell. used for debugging
kwarg_dict0 = dict(name='t', space='temporal', units='frame',
                  size=5, offset=0, scale=1)
kwarg_dict1 = dict(name='y', space='position', units='nm',
                  size=11, offset=-60, scale=10)
kwarg_dict2 = dict(name='x', space='position', units='nm',
                  size=21, offset=-60, scale=5)
kwarg_dict3 = dict(name='E', space='spectral', units='eV',
                  size=512, offset=-1.6, scale=0.1)

kwarg_dicts = [kwarg_dict0, kwarg_dict1, kwarg_dict2, kwarg_dict3]
dimensions = Dimensions(kwarg_dicts, nav_dimensions=[1,2], sig_dimensions=[3])

dimensions.show_tree()

## Indexing Dimension(s)

A dimensions values can be indexed directly for convenience.

In [ ]:
dimension = dimensions['x'].deepcopy()
dimension[0:300.]

Dimensions can be indexed by name or integerr

In [ ]:
print(dimensions.get_names())
display(dimensions[0])
display(dimensions['x'])

# Signal

In [ ]:
from pySEA.sea_eco.architecture.base_structure_numpy import Signal

Signal()

In [ ]:
from pySEA.sea_eco.test_dataset_generators import generate_gaussian_SI_t_series_data

sig_data = generate_gaussian_SI_t_series_data(dimensions=dimensions)

signal = Signal(sig_data, name='SI', dimensions=dimensions, metadata=meta,
                original_metadata=Metadata({'source':'simulated data'})
                )
signal.show_tree()

## Singal Indexing

Demonstrate indexing

In [ ]:
s = signal[0,-50.:30.:20.]
print(signal.dimensions.dimensions)
print(s.dimensions.dimensions)
print(signal.data.shape, s.data.shape)
s.dimensions

In [ ]:
print(f'original shape: {signal.data.shape}')
print(f'index [0,1:4] shape: {signal[0,1:4].data.shape}')
print(f'index [0,4.:36.:8.] shape: {signal[0,-10.:30.:8.].data.shape}')
print(f'index [0,...,-1] shape: {signal[0,...,-1].data.shape}')

## Funciton wrapping

If the funciton collapses the signal to 0-D then a value is returned not a signal

In [ ]:
s = np.sum(signal)
print(f'{signal.dimensions} to {s}')

Integer and tuple indexing are both valid

In [ ]:
s = np.sum(signal, axis=1)
print(s)
print(f'{signal.dimensions}({signal.data.shape}) \nto {s.dimensions}({s.data.shape})')
print()

s = np.sum(signal, axis=(1,2))
print(f'{signal.dimensions}({signal.data.shape}) \nto {s.dimensions}({s.data.shape})')

Both dimension names and reverese indexing are supported

In [ ]:
s = np.sum(signal, axis=('t',-1))
print(f'{signal.dimensions}({signal.data.shape}) \nto {s.dimensions}({s.data.shape})')

## Plotting

### Matplotlib figures

In [ ]:
import matplotlib.pyplot as plt

Matplotlib plotting is supported using `signal.show()`.<br />
Important general kwargs include:
* `ax`:matplotlib axis. Axis to plot to. If None then an axis is both created.
* `dims`: Specification of dimensions to plot. In this example we use 'nav' as an example.
* `fnc`: Any callable funciton that reduces the dimensions. In this example third pannel you can see that sum does not show anything but noise because the EELS loss-function takes signal from the zero-loss line, so the net change in signal is zero+noise. However, the standard deviation of the signal changes dramatically where there is loss, as shown in pannel 4.

In [ ]:
print('Signal dimensions:', signal.dimensions)

fig,axs=plt.subplots(ncols=4)
np.log10(signal).show(ax=axs[0], title='Default')
signal[0,...,0.].show(ax=axs[1], title='Sliced to 2D', ticks_and_labels='on', scale_bar=False)
signal.show(ax=axs[2], dims='nav', title='Navigation', aspect='auto')
signal.show(ax=axs[3], dims='nav', title='Navigation (fnc=std)', aspect='auto', fnc=np.std)
fig.tight_layout()


### Interactive

In [ ]:
from pySEA.sea_eco._plotting.interactive.ipywidget import interactive_signal_plot

The default note book interactive plotting provides navigator display dropdowns and signal dropdowns to select what will be plotted. Navigator sliders appear for all non-signal axes to aid in navigation.

In [ ]:
interactive_signal_plot(signal, nav_fnc=np.std)

The initial plotting dimension can also be provided and the selector turned off. Integer, negative, and named dimension refferences are all supported.

In [ ]:
interactive_signal_plot(signal,
                      nav_dimensions=[0, -1],
                      sig_dimensions=['y','x'],
                      show_dimension_selector=False)

The function used to collapse navigation axes can be supplied.

In [ ]:
interactive_signal_plot(signal, nav_fnc=np.std)

## Other

The signals can be unfolded to iterate over axes.

In [ ]:
state = signal.unfold_axes(keep_axes=['x','E'])
state, signal.data.shape

They can then be refolded

In [ ]:
signal.fold_axes()
signal.data.shape

# SignalSet


A `SignalSet` is intended for multimodal datasets that share similar metadata and dimensions. The `SignalSet` class therefore merges the `Metadata` and `Dimensions` injected attributes promoting them from signal level to set level. The `SignalSet` level attributes are now shared for the signals, and if for example one of the `SignalSet` dimensions is calibrated, then the calibration will be reflected for all contained signals.

In [ ]:
from pySEA.sea_eco.architecture.base_structure_numpy import SignalSet

display('SignalSet')
SignalSet().show_tree()

Generate some more data to add to the set...otherwise this is boring.

In [ ]:
adf = signal.deepcopy_with_reduced_data_dim(-0.1*np.sum(signal[...,:5.], axis=-1), keep_dim=(0,1,2))
adf.metadata.Instrument.Detectors.update_from_dict({'ADF':{'exposure':10.}})
del adf.metadata.Instrument.Detectors.Dectris_ELA

abf = signal.deepcopy_with_reduced_data_dim(np.sum(signal[...,:5.], axis=-1), keep_dim=(0,1,2))
abf.metadata.Instrument.Detectors.update_from_dict({'ABF':{'exposure':10.}})
del abf.metadata.Instrument.Detectors.Dectris_ELA

signals = [adf, abf, signal]
names = ['ADF', 'vBF', 'SI']
for signal,name in zip(signals,names): signal.name=name
print('Signals in set:')
pprint(signals)

In [ ]:
display('SignalSet')
signal_set = SignalSet(signals=signals, name='SI scan 1', main_signal=-1)
signal_set.show_tree()

# SignalColelctions

A `SignalColelction` is much like a `SignalSet` as it is fundamentally another functionalized container. Like `Signal` and `SignalSet`, `SignalColelction` can also have metadata to describe the `SignalColelction` and its contents. `SignalColelction` is however not intended to have shared dimensions and can contain either `Signal` or `SignalSet`.

In [ ]:
from pySEA.sea_eco.architecture.base_structure_numpy import SignalCollection

SignalCollection()

In [ ]:
signal_collection = SignalCollection(datasets=[signal, signal_set], name='Collection 1')
signal_collection.show_tree()

In [ ]:
signal_collection['SI'].show_tree()

# SEAFile

A SEAFile contains <u>S</u>imulation <u>E</u>xperiment and <u>A</u>nalysis collections, which allows for the structured and deleniated comparison of data.

In [ ]:
from pySEA.sea_eco.architecture.base_structure_numpy import SEAFile

SEAFile()

The SEAFile SEA collections can be initialized by directly providing a SiganlCollection or Sequence of Signals, SignalSets, and/or SignalCollections.

In [ ]:
sea_file = SEAFile(name='SEA file name',
                   experiments=signal_collection, 
                   simulations=[signal],
                   metadata={'key':'file level metadata'})
sea_file.show_tree()

In [ ]:
signal_set2 = signal_set.deepcopy()
signal_set2.name = 'SI scan 2'
sea_file.add_experiment(signal_set2)

sea_file.show_tree()

# Shared SEASerializable features

There are many features that are shared by all SEASerializable classes. These features, for example, aid in:
1) metadata searching
2) provenance tracking
3) IO

## Attribute Search

In [ ]:
signal_set.find_attribute('uuid', include_hidden=False)

## Save and load

Any `SEASerialized` class can be written to a SEA file format. This file format is built uspon HDF5 and can be viewed or loaded with external software.

In [ ]:
meta.to_sea('./sea_files/meta.sea')
dimension.to_sea('./sea_files/dimension.sea')
dimensions.to_sea('./sea_files/dimensions.sea')
signal.to_sea('./sea_files/signal.sea')
signal_set.to_sea('./sea_files/signal_set.sea')
signal_collection.to_sea('./sea_files/signal_collection.sea')
sea_file.to_sea('./sea_files/sea_file.sea')

Any `SEASerialized` calss can also be read using the from_sea class method or the IO load function.

In [ ]:
sea_file_loaded = SEAFile()
sea_file_loaded.from_sea('./sea_files/sea_file.sea')
sea_file_loaded.show_tree()

In [ ]:
from pySEA.sea_eco.io import load

load(file_path=r'./sea_files/sea_file.sea').show_tree()